# Non-graph baselines: Cosine similarity + MLP-only

Two simple baselines that **do not use the citation graph at all**. Both use the same author representation (mean-pool of pre-year history paper embeddings), but score with raw features only:

1. **Cosine similarity baseline** — no learning. For author $u$ and candidate paper $v$, score $s(u,v) = \cos(\bar X_{H_u},\, X_v)$ where $X \in \mathbb{R}^{N_p \times 768}$ is the raw title-embedding matrix and $\bar X_{H_u}$ is the mean of $u$'s history rows.
2. **MLP-only baseline** — same author rep, but score with a trained 2-layer MLP decoder operating on raw 768-dim features. No GNN.

**Why these baselines matter for the paper.** The headline claim *"the citation graph adds value"* is only credible if non-graph methods do worse on the same eval. These two baselines bracket the no-graph performance: cosine = no-learning floor, MLP = best non-graph learner.

**Both eval regimes are computed:** 1K mixed pool and full pool ($\sim 546$K candidates), exactly the same as in the GNN notebooks, for apples-to-apples comparison.

## Imports + speed flags

In [1]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from pathlib import Path

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

## Reproducibility, device, GPU info

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA B200
VRAM: 191.5 GB


## Configuration

MLP decoder mirrors GraphSAGE's decoder shape but takes 768-dim features instead of 128-dim GNN outputs: $\text{Linear}(2 \cdot 768 \to 128) \to \text{ReLU} \to \text{Linear}(128 \to 1)$. Same training schedule as v2 GraphSAGE (30 epochs, batch 1024, mixed negatives). Bf16 autocast for the MLP forward.

In [3]:
PROCESSED_DIR = Path("./data/processed_v2")
MODEL_OUT_DIR = Path("./models")
PLOTS_OUT_DIR = Path("./data")

NEG_STRATEGY = 'mixed'
FEATURE_DIM = 768
DECODER_HIDDEN_DIM = 128

LR = 1e-3
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 30
BATCH_SIZE = 1024
EVAL_EVERY = 5
EVAL_AUTHOR_BATCH = 256        # 1K mixed batched
EVAL_FULLPOOL_BATCH = 16       # full pool batched (memory-tight)
USE_BFLOAT16 = True

print(f"Data dir:      {PROCESSED_DIR}")
print(f"NEG_STRATEGY:  {NEG_STRATEGY}")
print(f"MLP decoder:   Linear(2*{FEATURE_DIM} -> {DECODER_HIDDEN_DIM}) -> ReLU -> Linear({DECODER_HIDDEN_DIM} -> 1)")
print(f"MLP training:  {NUM_EPOCHS} epochs, batch {BATCH_SIZE}, eval every {EVAL_EVERY}")
print(f"bfloat16:      {USE_BFLOAT16}")

Data dir:      data/processed_v2
NEG_STRATEGY:  mixed
MLP decoder:   Linear(2*768 -> 128) -> ReLU -> Linear(128 -> 1)
MLP training:  30 epochs, batch 1024, eval every 5
bfloat16:      True


## Step 1 — Load data

Same loads as the GNN notebooks. We don't need any graph edge information; we only use `hetero_graph['paper'].x` (the raw 768-dim title embeddings, identical to what the GNN encoders see as input).

In [4]:
train_years = torch.load(PROCESSED_DIR / "train_years.pt", weights_only=False)
metadata = torch.load(PROCESSED_DIR / "metadata.pt", weights_only=False)
val_data = torch.load(PROCESSED_DIR / "val.pt", weights_only=False)
test_data = torch.load(PROCESSED_DIR / "test.pt", weights_only=False)

print(f"Training years ({len(train_years)}): {train_years}")
print(f"Val examples:  {len(val_data['supervision']):,}")
print(f"Test examples: {len(test_data['supervision']):,}")
assert metadata['embed_dim'] == FEATURE_DIM

Training years (12): [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016]
Val examples:  43,562
Test examples: 37,046


## Step 2 — Preload year features on GPU + pre-tensorize supervision

We only need the **paper feature matrix** $X_y$ for each year, not the graph edges. Each $X_y$ is $(N_p^{(y)}, 768)$ — papers with year $< y$. Total memory: same $\sim 36$ GB across all years as the GNN notebooks.

Each supervision example's paper-id lists are pre-tensorized to GPU once, eliminating per-batch allocations.

In [5]:
def tensorize_examples(supervision, mode, device):
    for ex in supervision:
        ex['_hist_t'] = torch.tensor(ex['history_locals'], dtype=torch.long, device=device)
        ex['_pos_t']  = torch.tensor(ex['positive_locals'], dtype=torch.long, device=device)
        if mode == 'train':
            ex['_negr_t'] = torch.tensor(ex['negatives_random'], dtype=torch.long, device=device)
            ex['_negh_t'] = torch.tensor(ex['negatives_hard'], dtype=torch.long, device=device)
        else:
            ex['_negm_t'] = torch.tensor(ex['negative_locals_mixed'], dtype=torch.long, device=device)
            if 'known_locals' in ex:
                ex['_known_t'] = torch.tensor(list(ex['known_locals']), dtype=torch.long, device=device)
                ex['_pos_set'] = set(ex['positive_locals'])

print("Pre-loading paper features for all training years...")
all_year_X = {}      # year -> (N_p^y, 768) feature matrix on device
all_year_sup = {}    # year -> list of supervision examples (with pre-tensorized fields)
t0 = time.time()
for year in train_years:
    yd = torch.load(PROCESSED_DIR / f"train_year_{year}.pt", weights_only=False)
    X_y = yd['hetero_graph']['paper'].x.to(device)  # (N_p^y, 768)
    tensorize_examples(yd['supervision'], mode='train', device=device)
    all_year_X[year] = X_y
    all_year_sup[year] = yd['supervision']
    print(f"  year {year}: X shape {tuple(X_y.shape)}, {len(yd['supervision']):,} examples")
print(f"Loaded {len(all_year_X)} years in {time.time()-t0:.1f}s")

print("\nPreparing val + test features and tensorizing...")
val_X = val_data['hetero_graph']['paper'].x.to(device)
test_X = test_data['hetero_graph']['paper'].x.to(device)
tensorize_examples(val_data['supervision'], mode='eval', device=device)
tensorize_examples(test_data['supervision'], mode='eval', device=device)
print(f"Val X: {tuple(val_X.shape)},| Test X: {tuple(test_X.shape)}")

if torch.cuda.is_available():
    print(f"\nGPU memory after preload: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

Pre-loading paper features for all training years...
  year 2005: X shape (107163, 768), 14,780 examples
  year 2006: X shape (125063, 768), 16,944 examples
  year 2007: X shape (144229, 768), 19,955 examples
  year 2008: X shape (165790, 768), 22,485 examples
  year 2009: X shape (189419, 768), 26,268 examples
  year 2010: X shape (215312, 768), 29,597 examples
  year 2011: X shape (243744, 768), 32,787 examples
  year 2012: X shape (274267, 768), 36,438 examples
  year 2013: X shape (307407, 768), 39,737 examples
  year 2014: X shape (342789, 768), 42,423 examples
  year 2015: X shape (379003, 768), 45,249 examples
  year 2016: X shape (418317, 768), 45,364 examples
Loaded 12 years in 31.3s

Preparing val + test features and tensorizing...
Val X: (456776, 768),| Test X: (495848, 768)

GPU memory after preload: 13.6 GB


## Step 3 — Common helpers (mean-pool, BPR, neg helper)

Identical to v2 GNN notebooks. The mean-pool author embedding is computed from the **raw paper feature matrix** $X$ (no GNN). For both baselines.

In [6]:
def compute_author_embeddings(X, supervision_examples):
    """Author = mean of pre-year history paper rows of X. X has shape (N_p, d)."""
    return torch.stack([X[ex['_hist_t']].mean(dim=0) for ex in supervision_examples])

def bpr_loss(pos_scores, neg_scores):
    diff = pos_scores.unsqueeze(1) - neg_scores.unsqueeze(0)
    return -F.logsigmoid(diff).mean()

def get_train_negatives_t(ex, strategy):
    if strategy == 'random':
        return ex['_negr_t']
    if strategy == 'hard':
        return ex['_negh_t']
    n = ex['_negr_t'].shape[0]
    half = n // 2
    return torch.cat([ex['_negr_t'][:half], ex['_negh_t'][:n - half]])

## Baseline 1 — Cosine similarity (no learning)

For author $u$ and candidate $v$:
$$s_{\cos}(u, v) \;=\; \frac{\bar X_{H_u} \cdot X_v}{\|\bar X_{H_u}\|\,\|X_v\|}.$$

No parameters. We just $\ell_2$-normalize the author embedding and the candidate features, then dot-product. **One matmul per author** $\to$ trivially fast on GPU.

Two evaluation paths share helper logic:

- **1K mixed**: build padded candidate tensor $(M, L_{\max})$, gather feature rows, compute cosine, sort, find best positive rank.
- **Full pool**: pre-normalize the entire $X$ once, then for each author batch compute $A \cdot X^\top$ in one shot, mask known papers to $-\infty$, sort.

The full-pool case is essentially **one big matmul of shape $(M_{\text{batch}} \times 546\text{K})$**. On B200, the entire test set (37K authors × 546K papers) finishes in seconds.

### Cosine 1K-mixed eval

Identical structure to the GNN `evaluate_fast`, except the score is `cosine_similarity` instead of `decoder(...)`.

In [7]:
@torch.no_grad()
def cosine_evaluate_1k(X, eval_data, device, author_batch_size=256):
    examples = eval_data['supervision']
    if not examples:
        return {'hits@10': 0.0, 'mrr': 0.0, 'num_authors': 0}

    M = len(examples)
    # Author embs (raw mean-pool)
    author_embs = torch.stack([X[ex['_hist_t']].mean(dim=0) for ex in examples])  # (M, d)
    # Pre-normalize (once)
    a_norm = F.normalize(author_embs, dim=-1)
    X_norm = F.normalize(X, dim=-1)                                                # (N_p, d)

    # Build padded candidate IDs and masks (same as GNN eval)
    n_pos_arr = [ex['_pos_t'].shape[0] for ex in examples]
    cand_t_list = [torch.cat([ex['_pos_t'], ex['_negm_t']]) for ex in examples]
    cand_lens = [c.shape[0] for c in cand_t_list]
    max_len = max(cand_lens)

    cand_ids = torch.zeros(M, max_len, dtype=torch.long, device=device)
    valid = torch.zeros(M, max_len, dtype=torch.bool, device=device)
    pos_mask = torch.zeros(M, max_len, dtype=torch.bool, device=device)
    for i, (c, np_) in enumerate(zip(cand_t_list, n_pos_arr)):
        L = c.shape[0]
        cand_ids[i, :L] = c
        valid[i, :L] = True
        pos_mask[i, :np_] = True

    hits10 = torch.zeros(M, device=device)
    mrrs = torch.zeros(M, device=device)

    for bs in range(0, M, author_batch_size):
        be = min(bs + author_batch_size, M)
        a = a_norm[bs:be]                                  # (b, d)
        cids = cand_ids[bs:be]                              # (b, L)
        cembs = X_norm[cids]                                 # (b, L, d)
        # Cosine similarity = sum over d of a_b * c_{b,L,d}
        scores = (a.unsqueeze(1) * cembs).sum(dim=-1)        # (b, L)
        scores = scores.masked_fill(~valid[bs:be], float('-inf'))

        _, ranked = scores.sort(dim=-1, descending=True)
        is_pos_at_rank = torch.gather(pos_mask[bs:be], 1, ranked)
        any_pos = is_pos_at_rank.any(dim=-1)
        first_pos = is_pos_at_rank.float().argmax(dim=-1)
        best_rank = torch.where(any_pos, first_pos + 1, torch.tensor(max_len + 1, device=device))
        hits10[bs:be] = (best_rank <= 10).float()
        mrrs[bs:be] = torch.where(any_pos, 1.0 / best_rank.float(), torch.zeros_like(best_rank.float()))

    return {'hits@10': float(hits10.mean()), 'mrr': float(mrrs.mean()), 'num_authors': M}

### Cosine full-pool eval

Pre-normalize $X$ once. For each author batch, compute $A \cdot X^\top$ in one matmul. Mask known papers minus positives. Find best positive rank.

Memory: a batch of $b$ authors produces a $b \times N_p$ score matrix. With $b = 256$, $N_p = 546$K, fp32: $\sim 560$ MB. Easy.

In [8]:
@torch.no_grad()
def cosine_evaluate_full(X, eval_data, device, author_batch_size=256):
    examples = eval_data['supervision']
    if not examples:
        return {'hits@10': 0.0, 'mrr': 0.0, 'num_authors': 0}
    if '_known_t' not in examples[0]:
        raise KeyError("Full-pool eval requires 'known_locals'.")

    M = len(examples)
    N = X.shape[0]
    author_embs = torch.stack([X[ex['_hist_t']].mean(dim=0) for ex in examples])   # (M, d)
    a_norm = F.normalize(author_embs, dim=-1)
    X_norm = F.normalize(X, dim=-1)                                                 # (N_p, d)

    hits10 = torch.zeros(M, device=device)
    mrrs = torch.zeros(M, device=device)

    for bs in range(0, M, author_batch_size):
        be = min(bs + author_batch_size, M)
        a = a_norm[bs:be]                            # (b, d)
        scores = a @ X_norm.T                         # (b, N_p)
        for j, gi in enumerate(range(bs, be)):
            ex = examples[gi]
            known_t = ex['_known_t']
            pos_set = ex['_pos_set']
            if known_t.numel():
                # Mask known papers that are NOT positives
                mask_idx = known_t[~torch.tensor([k.item() in pos_set for k in known_t], device=device)]
                if mask_idx.numel() > 0:
                    scores[j, mask_idx] = float('-inf')

        for j, gi in enumerate(range(bs, be)):
            ex = examples[gi]
            pos_t = ex['_pos_t']
            row = scores[j]
            _, ranked = row.sort(descending=True)
            in_pos = torch.isin(ranked, pos_t)
            nz = in_pos.nonzero()
            if nz.numel() > 0:
                best = nz[0].item() + 1
                hits10[gi] = 1.0 if best <= 10 else 0.0
                mrrs[gi] = 1.0 / best
    return {'hits@10': float(hits10.mean()), 'mrr': float(mrrs.mean()), 'num_authors': M}

### Run cosine baseline on test

Two numbers: 1K mixed Hits@10 and full-pool Hits@10. Both should beat random by a wide margin (the title features are highly clustered, so mean-pool similarity is informative). Cosine is also a hard floor for any learned non-graph model: if MLP doesn't beat cosine, learning over raw features wasn't useful.

In [9]:
print("Cosine baseline on test set...")
t0 = time.time()
cos_test_1k = cosine_evaluate_1k(test_X, test_data, device, EVAL_AUTHOR_BATCH)
t_1k = time.time() - t0
t0 = time.time()
cos_test_full = cosine_evaluate_full(test_X, test_data, device, EVAL_AUTHOR_BATCH)
t_full = time.time() - t0

print("=" * 60)
print("COSINE SIMILARITY BASELINE (no learning) — TEST")
print("=" * 60)
print(f"\n[1K mixed pool, 50/50 hard/random]  ({t_1k:.1f}s)")
print(f"  Hits@10: {cos_test_1k['hits@10']:.4f}")
print(f"  MRR:     {cos_test_1k['mrr']:.4f}")
print(f"\n[Full pool — every paper in the pre-test graph]  ({t_full:.1f}s)")
print(f"  Hits@10: {cos_test_full['hits@10']:.4f}")
print(f"  MRR:     {cos_test_full['mrr']:.4f}")

Cosine baseline on test set...
COSINE SIMILARITY BASELINE (no learning) — TEST

[1K mixed pool, 50/50 hard/random]  (1.3s)
  Hits@10: 0.1994
  MRR:     0.0961

[Full pool — every paper in the pre-test graph]  (56.5s)
  Hits@10: 0.0056
  MRR:     0.0036


## Baseline 2 — MLP-only (trained, no GNN)

Same author rep, but score with a 2-layer MLP that takes raw 768-dim author and paper features as input:
$$s_{\text{MLP}}(u, v) \;=\; W_2 \cdot \text{ReLU}\!\big(W_1 \cdot [\bar X_{H_u} \concat X_v] + b_1\big) + b_2,$$
where $W_1 \in \mathbb{R}^{128 \times 1536}$ (so $|W_1| = 196{,}608$), $W_2 \in \mathbb{R}^{1 \times 128}$. Total trainable parameters $\approx 196{,}737$.

Trained with BPR loss using the same supervision as every other model. **No GNN encode step — just per-batch MLP forward + backward.** Per-epoch wall-clock should be tiny (the encoder forward over 546K nodes was the dominant cost in the GNN models).

In [10]:
class MLPDecoder(nn.Module):
    def __init__(self, feat_dim, hidden_dim, dropout=0.0):
        super().__init__()
        self.fc1 = nn.Linear(2 * feat_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)
        self.dropout = dropout

    def forward(self, author_emb, paper_emb):
        combined = torch.cat([author_emb, paper_emb], dim=-1)
        h = F.relu(self.fc1(combined))
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.fc2(h).squeeze(-1)

decoder = MLPDecoder(FEATURE_DIM, DECODER_HIDDEN_DIM, dropout=0.0).to(device)
optimizer = torch.optim.Adam(decoder.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
n_dec = sum(p.numel() for p in decoder.parameters())
print(f"MLP decoder params: {n_dec:,}")

MLP decoder params: 196,865


### Train one year (MLP-only)

Same structure as the GNN training step but **no encoder forward** (we use raw features directly). All operations on the pre-loaded $X_y$ tensor, no `torch.tensor()` allocations in the hot loop.

In [11]:
def train_one_year_mlp(decoder, optimizer, X, examples, device,
                       batch_size=1024, neg_strategy='mixed', use_amp=True):
    decoder.train()
    if not examples:
        return 0.0
    amp_ctx = torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16) if use_amp else \
              torch.amp.autocast(device_type='cuda', enabled=False)

    indices = list(range(len(examples)))
    random.shuffle(indices)
    total_loss, num_batches = 0.0, 0

    for batch_start in range(0, len(indices), batch_size):
        bex = [examples[i] for i in indices[batch_start:batch_start + batch_size]]
        # Author embeddings = mean-pool of raw history rows
        author_embs = torch.stack([X[ex['_hist_t']].mean(dim=0) for ex in bex])  # (B, d)

        all_a, all_p, bounds = [], [], []
        offset = 0
        for i, ex in enumerate(bex):
            negs_t = get_train_negatives_t(ex, neg_strategy)
            if negs_t.numel() == 0:
                continue
            n_pos, n_neg = ex['_pos_t'].shape[0], negs_t.shape[0]
            a = author_embs[i]
            all_a.append(a.unsqueeze(0).expand(n_pos + n_neg, -1))
            all_p.append(torch.cat([X[ex['_pos_t']], X[negs_t]], dim=0))
            bounds.append((offset, n_pos, n_neg))
            offset += n_pos + n_neg
        if not all_a:
            continue

        flat_a = torch.cat(all_a, dim=0)
        flat_p = torch.cat(all_p, dim=0)

        with amp_ctx:
            scores = decoder(flat_a, flat_p)
            batch_loss = torch.tensor(0.0, device=device)
            for start, n_pos, n_neg in bounds:
                ps = scores[start:start + n_pos]
                ns = scores[start + n_pos:start + n_pos + n_neg]
                batch_loss = batch_loss + bpr_loss(ps, ns)
            batch_loss = batch_loss / len(bounds)

        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()
        total_loss += float(batch_loss.detach())
        num_batches += 1

    return total_loss / max(num_batches, 1)

### MLP eval (1K mixed) — vectorized

In [12]:
@torch.no_grad()
def mlp_evaluate_1k(decoder, X, eval_data, device, author_batch_size=256):
    decoder.eval()
    examples = eval_data['supervision']
    if not examples:
        return {'hits@10': 0.0, 'mrr': 0.0, 'num_authors': 0}

    M = len(examples)
    d = X.shape[1]
    author_embs = torch.stack([X[ex['_hist_t']].mean(dim=0) for ex in examples])

    n_pos_arr = [ex['_pos_t'].shape[0] for ex in examples]
    cand_t_list = [torch.cat([ex['_pos_t'], ex['_negm_t']]) for ex in examples]
    max_len = max(c.shape[0] for c in cand_t_list)

    cand_ids = torch.zeros(M, max_len, dtype=torch.long, device=device)
    valid = torch.zeros(M, max_len, dtype=torch.bool, device=device)
    pos_mask = torch.zeros(M, max_len, dtype=torch.bool, device=device)
    for i, (c, np_) in enumerate(zip(cand_t_list, n_pos_arr)):
        L = c.shape[0]
        cand_ids[i, :L] = c
        valid[i, :L] = True
        pos_mask[i, :np_] = True

    hits10 = torch.zeros(M, device=device)
    mrrs = torch.zeros(M, device=device)

    for bs in range(0, M, author_batch_size):
        be = min(bs + author_batch_size, M)
        b = be - bs
        a = author_embs[bs:be]                    # (b, d)
        cids = cand_ids[bs:be]                     # (b, L)
        cembs = X[cids]                             # (b, L, d)
        a_exp = a.unsqueeze(1).expand(-1, max_len, -1)
        flat_a = a_exp.reshape(-1, d)
        flat_c = cembs.reshape(-1, d)
        scores = decoder(flat_a, flat_c).reshape(b, max_len)
        scores = scores.masked_fill(~valid[bs:be], float('-inf'))

        _, ranked = scores.sort(dim=-1, descending=True)
        is_pos_at_rank = torch.gather(pos_mask[bs:be], 1, ranked)
        any_pos = is_pos_at_rank.any(dim=-1)
        first_pos = is_pos_at_rank.float().argmax(dim=-1)
        best_rank = torch.where(any_pos, first_pos + 1, torch.tensor(max_len + 1, device=device))
        hits10[bs:be] = (best_rank <= 10).float()
        mrrs[bs:be] = torch.where(any_pos, 1.0 / best_rank.float(), torch.zeros_like(best_rank.float()))

    return {'hits@10': float(hits10.mean()), 'mrr': float(mrrs.mean()), 'num_authors': M}

### MLP eval (full pool) — author-batched, masking known papers

In [13]:
@torch.no_grad()
def mlp_evaluate_full(decoder, X, eval_data, device, author_batch_size=16):
    decoder.eval()
    examples = eval_data['supervision']
    if not examples:
        return {'hits@10': 0.0, 'mrr': 0.0, 'num_authors': 0}
    if '_known_t' not in examples[0]:
        raise KeyError("Full-pool eval requires 'known_locals'.")

    M = len(examples)
    N, d = X.shape
    author_embs = torch.stack([X[ex['_hist_t']].mean(dim=0) for ex in examples])
    hits10 = torch.zeros(M, device=device)
    mrrs = torch.zeros(M, device=device)

    for bs in range(0, M, author_batch_size):
        be = min(bs + author_batch_size, M)
        b = be - bs
        a = author_embs[bs:be]                                 # (b, d)
        a_exp = a.unsqueeze(1).expand(-1, N, -1).reshape(-1, d)
        p_exp = X.unsqueeze(0).expand(b, -1, -1).reshape(-1, d)
        scores = decoder(a_exp, p_exp).reshape(b, N)            # (b, N)

        for j, gi in enumerate(range(bs, be)):
            ex = examples[gi]
            known_t = ex['_known_t']
            pos_set = ex['_pos_set']
            if known_t.numel():
                mask_idx = known_t[~torch.tensor([k.item() in pos_set for k in known_t], device=device)]
                if mask_idx.numel() > 0:
                    scores[j, mask_idx] = float('-inf')

        for j, gi in enumerate(range(bs, be)):
            ex = examples[gi]
            pos_t = ex['_pos_t']
            row = scores[j]
            _, ranked = row.sort(descending=True)
            in_pos = torch.isin(ranked, pos_t)
            nz = in_pos.nonzero()
            if nz.numel() > 0:
                best = nz[0].item() + 1
                hits10[gi] = 1.0 if best <= 10 else 0.0
                mrrs[gi] = 1.0 / best

    return {'hits@10': float(hits10.mean()), 'mrr': float(mrrs.mean()), 'num_authors': M}

### MLP training loop

30 epochs, evaluate val every 5 epochs (1K mixed pool only — full pool too slow during training, we'll only do it at test). Tracks best-val-MRR weights.

In [14]:
train_losses, val_hits, val_mrrs, eval_epochs = [], [], [], []
best_val_mrr = -1.0
best_epoch = 0
best_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    t_epoch = time.time()
    epoch_losses = []
    for year in train_years:
        l = train_one_year_mlp(
            decoder, optimizer, all_year_X[year], all_year_sup[year], device,
            batch_size=BATCH_SIZE, neg_strategy=NEG_STRATEGY, use_amp=USE_BFLOAT16,
        )
        epoch_losses.append(l)
    avg = float(np.mean(epoch_losses))
    train_losses.append(avg)
    epoch_dur = time.time() - t_epoch

    if epoch % EVAL_EVERY == 0 or epoch == NUM_EPOCHS:
        t_eval = time.time()
        v = mlp_evaluate_1k(decoder, val_X, val_data, device, EVAL_AUTHOR_BATCH)
        eval_dur = time.time() - t_eval
        val_hits.append(v['hits@10']); val_mrrs.append(v['mrr']); eval_epochs.append(epoch)

        improved = v['mrr'] > best_val_mrr
        if improved:
            best_val_mrr = v['mrr']
            best_epoch = epoch
            best_state = {k: vv.detach().clone() for k, vv in decoder.state_dict().items()}
        marker = " *" if improved else ""
        print(f"Epoch {epoch:>3}/{NUM_EPOCHS} | Loss {avg:.4f} | "
              f"Val H@10 {v['hits@10']:.4f} | Val MRR {v['mrr']:.4f}{marker} | "
              f"train {epoch_dur:.1f}s, eval {eval_dur:.1f}s")
    else:
        print(f"Epoch {epoch:>3}/{NUM_EPOCHS} | Loss {avg:.4f} | train {epoch_dur:.1f}s")

Epoch   1/30 | Loss 0.6188 | train 43.4s
Epoch   2/30 | Loss 0.5196 | train 44.0s
Epoch   3/30 | Loss 0.4939 | train 43.2s
Epoch   4/30 | Loss 0.4830 | train 45.5s
Epoch   5/30 | Loss 0.4765 | Val H@10 0.4394 | Val MRR 0.2121 * | train 43.4s, eval 1.5s
Epoch   6/30 | Loss 0.4710 | train 43.5s
Epoch   7/30 | Loss 0.4679 | train 45.7s
Epoch   8/30 | Loss 0.4646 | train 43.6s
Epoch   9/30 | Loss 0.4619 | train 43.2s
Epoch  10/30 | Loss 0.4593 | Val H@10 0.4513 | Val MRR 0.2200 * | train 42.8s, eval 1.5s
Epoch  11/30 | Loss 0.4579 | train 45.1s
Epoch  12/30 | Loss 0.4560 | train 42.9s
Epoch  13/30 | Loss 0.4547 | train 43.3s
Epoch  14/30 | Loss 0.4539 | train 42.9s
Epoch  15/30 | Loss 0.4533 | Val H@10 0.4646 | Val MRR 0.2227 * | train 45.0s, eval 1.5s
Epoch  16/30 | Loss 0.4520 | train 42.7s
Epoch  17/30 | Loss 0.4517 | train 43.5s
Epoch  18/30 | Loss 0.4510 | train 43.7s
Epoch  19/30 | Loss 0.4506 | train 45.7s
Epoch  20/30 | Loss 0.4509 | Val H@10 0.4651 | Val MRR 0.2264 * | train 43.3s

### Restore best weights, run MLP test

In [15]:
if best_state is not None:
    decoder.load_state_dict(best_state)
    print(f"Restored best MLP checkpoint (epoch {best_epoch}, val MRR {best_val_mrr:.4f})")

t0 = time.time()
mlp_test_1k = mlp_evaluate_1k(decoder, test_X, test_data, device, EVAL_AUTHOR_BATCH)
t_1k = time.time() - t0
t0 = time.time()
mlp_test_full = mlp_evaluate_full(decoder, test_X, test_data, device, EVAL_FULLPOOL_BATCH)
t_full = time.time() - t0

print("=" * 60)
print(f"MLP-ONLY BASELINE (no GNN) — TEST  (best epoch {best_epoch})")
print("=" * 60)
print(f"\n[1K mixed pool, 50/50 hard/random]  ({t_1k:.1f}s)")
print(f"  Hits@10: {mlp_test_1k['hits@10']:.4f}")
print(f"  MRR:     {mlp_test_1k['mrr']:.4f}")
print(f"\n[Full pool — every paper in the pre-test graph]  ({t_full:.1f}s)")
print(f"  Hits@10: {mlp_test_full['hits@10']:.4f}")
print(f"  MRR:     {mlp_test_full['mrr']:.4f}")

Restored best MLP checkpoint (epoch 25, val MRR 0.2300)
MLP-ONLY BASELINE (no GNN) — TEST  (best epoch 25)

[1K mixed pool, 50/50 hard/random]  (1.3s)
  Hits@10: 0.4885
  MRR:     0.2386

[Full pool — every paper in the pre-test graph]  (198.6s)
  Hits@10: 0.0084
  MRR:     0.0048


## Summary: cosine vs MLP vs GNN models

Combined comparison table. Cosine is the no-learning floor; MLP is the best non-graph learner; GNN models (from other notebooks) are graph-aware. Any GNN that fails to beat MLP across both eval regimes is not adding value over feature-based learning.

In [16]:
print("=" * 72)
print("NON-GRAPH BASELINES — TEST RESULTS SUMMARY")
print("=" * 72)
print(f"\n{'Method':<20} {'1K H@10':>10} {'1K MRR':>10} {'Full H@10':>12} {'Full MRR':>10}")
print("-" * 72)
print(f"{'Cosine sim (no train)':<20} {cos_test_1k['hits@10']:>10.4f} {cos_test_1k['mrr']:>10.4f} "
      f"{cos_test_full['hits@10']:>12.4f} {cos_test_full['mrr']:>10.4f}")
print(f"{'MLP (no GNN)':<20} {mlp_test_1k['hits@10']:>10.4f} {mlp_test_1k['mrr']:>10.4f} "
      f"{mlp_test_full['hits@10']:>12.4f} {mlp_test_full['mrr']:>10.4f}")
print("-" * 72)
print(f"{'Reference: GNN models':<20}")
print(f"{'  GraphSAGE':<20} {0.5082:>10.4f} {0.2645:>10.4f} {0.0112:>12.4f} {0.0061:>10.4f}")
print(f"{'  GAT':<20} {0.6217:>10.4f} {0.3583:>10.4f} {0.0355:>12.4f} {0.0171:>10.4f}")
print(f"{'  HetGAT-4L (s42)':<20} {0.7372:>10.4f} {0.4846:>10.4f} {0.0763:>12.4f} {0.0349:>10.4f}")
print(f"{'  HetGAT-5L':<20} {0.7080:>10.4f} {0.4719:>10.4f} {0.0832:>12.4f} {0.0400:>10.4f}")

NON-GRAPH BASELINES — TEST RESULTS SUMMARY

Method                  1K H@10     1K MRR    Full H@10   Full MRR
------------------------------------------------------------------------
Cosine sim (no train)     0.1994     0.0961       0.0056     0.0036
MLP (no GNN)             0.4885     0.2386       0.0084     0.0048
------------------------------------------------------------------------
Reference: GNN models
  GraphSAGE              0.5082     0.2645       0.0112     0.0061
  GAT                    0.6217     0.3583       0.0355     0.0171
  HetGAT-4L (s42)        0.7372     0.4846       0.0763     0.0349
  HetGAT-5L              0.7080     0.4719       0.0832     0.0400


## Save baselines

Both result dicts saved to `models/baselines_cosine_mlp.pt` for reference in the report.

In [17]:
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)
save_path = MODEL_OUT_DIR / "baselines_cosine_mlp.pt"
torch.save({
    'cosine': {
        'test_1k': cos_test_1k,
        'test_full': cos_test_full,
    },
    'mlp': {
        'test_1k': mlp_test_1k,
        'test_full': mlp_test_full,
        'best_epoch': best_epoch,
        'val_best_mrr': best_val_mrr,
        'train_losses': train_losses,
        'val_hits': val_hits,
        'val_mrrs': val_mrrs,
        'eval_epochs': eval_epochs,
        'config': {
            'feature_dim': FEATURE_DIM,
            'decoder_hidden_dim': DECODER_HIDDEN_DIM,
            'lr': LR, 'weight_decay': WEIGHT_DECAY,
            'num_epochs': NUM_EPOCHS, 'batch_size': BATCH_SIZE,
            'neg_strategy': NEG_STRATEGY, 'use_bfloat16': USE_BFLOAT16,
            'seed': SEED,
        },
    },
    'data_version': 'v2',
    'fos_level': metadata.get('fos_level'),
}, save_path)
print(f"Saved {save_path}")

Saved models/baselines_cosine_mlp.pt
